In [ ]:
pip install torch transformers pandas openpyxl accelerate bitsandbytes

In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Load Model and Tokenizer locally on your 24 GB GPU
model_id = "Qwen/Qwen2.5-7B-Instruct"  # Use "Qwen/Qwen2.5-14B-Instruct" for higher capacity

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

# 2. Load your Excel sheet
# Ensure your excel has a column named 'roman_urdu_word'
input_excel_path = "D:\BORUE\final_qwen_expanded.xlsx" 
df = pd.read_excel(input_excel_path)

expanded_results = []

print("Generating query expansions...")
for index, row in df.iterrows():
    query_word = str(row['Common']).strip()
    
    # Prompt structured to force strict formatting matching your paper requirements
    prompt = f"""You are an expert cross-lingual information retrieval assistant. 
Given the Roman Urdu input word, provide query expansion terms.
Input Word: "{query_word}"

Provide your output strictly in the following format:
Roman Urdu: [1-2 words max]
English: [1-2 words max]
Nastaliq Urdu: [1-2 words max]

Do not include any extra introductory or concluding text."""

    messages = [
        {"role": "system", "content": "You are a helpful cross-lingual NLP assistant."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=64,
        temperature=0.1,  # Low temperature for deterministic, structured output
        do_sample=True
    )
    
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    # Store or parse output
    expanded_results.append({
        "input_roman_urdu": query_word,
        "baseline_expansion_output": response.strip()
    })

# 3. Save baseline outputs to a new excel sheet for your web URL retrieval evaluation
output_df = pd.DataFrame(expanded_results)
output_df.to_excel("baseline_expanded_queries.xlsx", index=False)
print("Baseline expansion complete! Saved to 'baseline_expanded_queries.xlsx'.")

In [ ]:
## Cell-1
#final  code with all guard rails
# This file holds code for generation of expansion by Qwen only
import os
import re
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Define paths as requested
input_dir = r"D:\BORUE"
input_file = os.path.join(input_dir, "input_queries.xlsx")
output_file = os.path.join(input_dir, "final_qwen_expanded.xlsx")
os.makedirs(input_dir, exist_ok=True)

# 2. Load Model & Tokenizer locally with 4-bit quantization (optimized for 24 GB GPU)
model_id = "Qwen/Qwen2.5-7B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

print("Loading tokenizer and model from local cache...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

# 3. Read input Excel file
print(f"Reading queries from: {input_file}")
df = pd.read_excel(input_file)

results = []

print("Generating query expansions with strict anti-drift guardrails...")
for index, row in df.iterrows():
    query_word = str(row['roman_urdu_word']).strip()
    
    # Rigorous prompt with explicit language definition and negative constraints
    prompt = f"""You are an expert cross-lingual information retrieval assistant specializing in South Asian low-resource languages. 
The input word provided is in **Roman Urdu** (the Urdu language written phonetically using the Latin/English alphabet).

Input Word: "{query_word}"

Task: Generate cross-lingual query expansion terms for this word, comprising:
- Up to 2 related Roman Urdu terms (Latin script)
- Up to 2 English translation/synonym terms
- Up to 2 Nastaliq Urdu script terms (Urdu alphabet)

CRITICAL INSTRUCTIONS:
1. Output ONLY the expansion terms separated by spaces on a single, continuous line.
2. Do NOT include any language labels, quotes, or markdown formatting.
3. STRICTLY AVOID Chinese, Japanese, Korean, or any other foreign characters. Use ONLY Latin script (English/Roman Urdu) and Arabic/Urdu script (Nastaliq)."""

    messages = [
        {"role": "system", "content": "You are a precise cross-lingual NLP and information retrieval assistant."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=48,
        temperature=0.1,  # Low temperature for deterministic, clean results
        do_sample=True
    )
    
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    
    # Post-processing guardrail: Strip out any accidental CJK or weird Unicode blocks
    cleaned_response = re.sub(r'[\u4e00-\u9fff\u3000-\u30ff\uff00-\uffef]+', '', response)
    cleaned_response = re.sub(r'\s+', ' ', cleaned_response).strip()
    
    results.append({
        "roman_urdu_word": query_word,
        "expanded_query": cleaned_response
    })

# 4. Save final output file
output_df = pd.DataFrame(results)
output_df.to_excel(output_file, index=False)
print(f"Success! Final clean baseline saved to: {output_file}")